# AEGIS-RF — гібридна байєсівська просторова ідентифікація Wi-Fi у КІІ
## Adversary-rEsistant Geolocation & Integrity for wi-fi Signals

Прикладний демонстраційний програмний проєкт **за Розділом 1** дисертаційного
дослідження: просторова ідентифікація джерел IEEE 802.11 у критичній інформаційній
інфраструктурі (КІІ) зі злиттям **RSSI-радіокарти** та **FTM/RTT-дальнометрії**,
**робастністю** до навмисних маніпуляцій (evil twin, deceptive ranging, deauth) і
**доказовою атрибуцією** (Spatial Attribution Record) для інтеграції із SOC/SIEM/SOAR.

> **Демонстраційні дані.** Усі числа характеризують синтетичну модель, параметри якої
> задані автором, а не реальні вимірювання. Проєкт ілюструє методологію та перевіряє
> відтворюваність обчислень.

**Реалізовані компоненти розділу:**
> 1.1 моделі загроз (rogue AP / evil twin / deauth) · 1.2 RSSI-fingerprinting і
> байєсівська локалізація · 1.3 FTM/RTT-дальнометрія · 1.4 гібридне злиття
> правдоподібностей і робастний інференс · 1.5 доказова атрибуція та SOC/SIEM.

## 0. Реконструкція пакета `src/` (самодостатня Colab-версія)

Ця версія **для Google Colab**: комірки нижче записують модулі пакета у `src/` просто у файловій системі середовища — жодних зовнішніх файлів чи `!git clone` не потрібно. Достатньо *Runtime → Run all*. Код ідентичний модульній версії з репозиторію, тож за `SEED = 80211` результати збігаються біт-у-біт.

> Модульна версія з `pytest`-тестами — у каталозі `aegis-rf/` репозиторію.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
open('src/__init__.py', 'w').close()
print('src/ готово')

In [ ]:
%%writefile src/environment.py
"""Синтетичне середовище КІІ: геометрія, зони, точки доступу, RSSI-радіокарта.

Synthetic critical-infrastructure (CII) indoor environment: geometry, zones,
access points and the RSSI radiomap.

Модель відтворює постановку Розділу 1: закрите приміщення КІІ з набором точок
доступу IEEE 802.11 (частина з яких підтримує FTM), виділеною **критичною зоною**
(напр. серверна) та регулярною сіткою опорних точок (RP-grid) для байєсівської
локалізації за RSSI-радіокартою. Радіокарта будується з детермінованої моделі
загасання log-distance — це «версіонований статистичний об'єкт» (п. 1.2.5).

Модуль залежить лише від :mod:`numpy`.
"""

from __future__ import annotations

import numpy as np

# ============================================================
# Геометрія приміщення / Room geometry (метри)
# ============================================================
AREA_W = 30.0  # ширина, м
AREA_H = 24.0  # глибина, м
GRID_STEP = 1.0  # крок RP-сітки радіокарти, м

# Точки доступу (anchors). Останні дві — всередині/біля критичної зони.
AP_POSITIONS = np.array(
    [
        [2.0, 2.0],
        [28.0, 2.0],
        [2.0, 22.0],
        [28.0, 22.0],
        [15.0, 12.0],
        [24.0, 18.0],
    ]
)
N_AP = len(AP_POSITIONS)

# Які AP підтримують FTM/RTT (802.11mc/az) — операційна асиметрія (п. 1.3.6).
FTM_CAPABLE = np.array([True, True, False, True, False, True])

# ============================================================
# Критична зона (напр. серверна) / Critical zone rectangle
# ============================================================
CRIT_X0, CRIT_X1 = 21.0, 29.0
CRIT_Y0, CRIT_Y1 = 15.0, 23.0

# ============================================================
# Параметри радіоканалу / Radio-channel parameters
# ============================================================
RSSI_P0 = -40.0  # опорна потужність на d0, дБм
RSSI_D0 = 1.0  # опорна відстань, м
RSSI_N = 3.0  # показник загасання (indoor)
RSSI_SIGMA = 4.0  # тіньове завмирання (shadowing), дБ
RSSI_SENSITIVITY = -92.0  # поріг чутливості приймача, дБм

FTM_SIGMA = 1.2  # СКВ похибки дальності FTM, м
FTM_NLOS_BIAS = 3.0  # середнє додатне зміщення NLOS, м
FTM_P_NLOS = 0.25  # частка NLOS-вимірювань

P_DROP_RSSI = 0.03  # ймовірність відсутності RSSI від AP
P_DROP_FTM = 0.15  # ймовірність відмови FTM-сеансу (енергетика/асиметрія)

_DIST_FLOOR = 0.3  # нижня межа відстані, щоб log10 не розходився


def build_grid(step: float = GRID_STEP) -> np.ndarray:
    """Регулярна RP-сітка кандидатних позицій, форма ``(G, 2)``."""
    xs = np.arange(step / 2, AREA_W, step)
    ys = np.arange(step / 2, AREA_H, step)
    gx, gy = np.meshgrid(xs, ys)
    return np.column_stack([gx.ravel(), gy.ravel()])


def in_critical(points: np.ndarray) -> np.ndarray:
    """Булева маска: чи належить точка критичній зоні. ``points`` форми ``(..., 2)``."""
    x, y = points[..., 0], points[..., 1]
    return (x >= CRIT_X0) & (x <= CRIT_X1) & (y >= CRIT_Y0) & (y <= CRIT_Y1)


def ap_distances(points: np.ndarray) -> np.ndarray:
    """Евклідові відстані від кожної точки до кожного AP → ``(P, N_AP)``."""
    diff = points[:, None, :] - AP_POSITIONS[None, :, :]
    dist = np.sqrt((diff**2).sum(-1))
    return np.maximum(dist, _DIST_FLOOR)


def pathloss_mean(dist: np.ndarray) -> np.ndarray:
    """Середній RSSI за log-distance моделлю (без завмирання), обрізаний знизу."""
    rssi = RSSI_P0 - 10.0 * RSSI_N * np.log10(dist / RSSI_D0)
    return np.maximum(rssi, RSSI_SENSITIVITY)


def build_radiomap(step: float = GRID_STEP):
    """Побудувати радіокарту. Повертає ``(grid, M, D_grid, crit_mask)``.

    * ``grid``      — ``(G, 2)`` координати RP;
    * ``M``         — ``(G, N_AP)`` середній RSSI у кожній RP для кожного AP;
    * ``D_grid``    — ``(G, N_AP)`` відстані RP→AP (для FTM-правдоподібності);
    * ``crit_mask`` — ``(G,)`` булева маска RP усередині критичної зони.
    """
    grid = build_grid(step)
    d_grid = ap_distances(grid)
    m = pathloss_mean(d_grid)
    crit_mask = in_critical(grid)
    return grid, m, d_grid, crit_mask

In [ ]:
%%writefile src/sensing.py
"""Генерація спостережень: RSSI-відбитки та FTM/RTT-дальності.

Observation generation: RSSI fingerprints and FTM/RTT ranges.

RSSI:  s_{i,a} = pathloss(d_{i,a}) + N(0, sigma^2);  AP недоступний з imовірністю
       P_DROP_RSSI або якщо сигнал нижчий за поріг чутливості.
FTM:   r_{i,a} = d_{i,a} + bias_NLOS + N(0, sigma_ftm^2);  доступний лише для
       FTM-сумісних AP і з imовірністю (1 - P_DROP_FTM).

Усі функції приймають генератор ``rng`` — це зберігає точний порядок відтворюваності.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from . import environment as env


@dataclass
class Observations:
    """Пакет спостережень для набору подій."""

    pos: np.ndarray  # (E, 2) істинні позиції джерел
    rssi: np.ndarray  # (E, N_AP) виміряний RSSI, дБм
    rssi_avail: np.ndarray  # (E, N_AP) доступність RSSI
    ftm: np.ndarray  # (E, N_AP) виміряна дальність FTM, м
    ftm_avail: np.ndarray  # (E, N_AP) доступність FTM
    in_crit: np.ndarray  # (E,) чи є істинна позиція в критичній зоні

    def copy(self) -> "Observations":
        return Observations(
            self.pos.copy(), self.rssi.copy(), self.rssi_avail.copy(),
            self.ftm.copy(), self.ftm_avail.copy(), self.in_crit.copy(),
        )


def generate_positions(rng: np.random.Generator, n: int, crit_fraction: float = 0.45) -> np.ndarray:
    """Згенерувати ``n`` істинних позицій; частка ``crit_fraction`` — у критичній зоні."""
    n_crit = int(round(n * crit_fraction))
    n_free = n - n_crit

    crit = np.column_stack(
        [
            rng.uniform(env.CRIT_X0, env.CRIT_X1, n_crit),
            rng.uniform(env.CRIT_Y0, env.CRIT_Y1, n_crit),
        ]
    )
    # Позиції поза критичною зоною: відбір із прийняттям (rejection sampling)
    free = np.empty((n_free, 2))
    filled = 0
    while filled < n_free:
        cand = np.column_stack(
            [rng.uniform(0, env.AREA_W, n_free), rng.uniform(0, env.AREA_H, n_free)]
        )
        ok = ~env.in_critical(cand)
        take = cand[ok][: n_free - filled]
        free[filled : filled + len(take)] = take
        filled += len(take)

    pos = np.vstack([crit, free])
    order = rng.permutation(n)  # перемішуємо, щоб зони не йшли блоками
    return pos[order]


def observe_rssi(rng: np.random.Generator, pos: np.ndarray):
    """Згенерувати RSSI-спостереження. Повертає ``(rssi, avail)`` форми ``(E, N_AP)``."""
    dist = env.ap_distances(pos)
    mean = env.pathloss_mean(dist)
    rssi = mean + rng.normal(0.0, env.RSSI_SIGMA, dist.shape)
    dropped = rng.random(dist.shape) < env.P_DROP_RSSI
    avail = (~dropped) & (rssi > env.RSSI_SENSITIVITY)
    return rssi, avail


def observe_ftm(rng: np.random.Generator, pos: np.ndarray):
    """Згенерувати FTM/RTT-дальності. Повертає ``(ftm, avail)`` форми ``(E, N_AP)``."""
    dist = env.ap_distances(pos)
    nlos = rng.random(dist.shape) < env.FTM_P_NLOS
    bias = nlos * rng.exponential(env.FTM_NLOS_BIAS, dist.shape)
    ftm = dist + bias + rng.normal(0.0, env.FTM_SIGMA, dist.shape)
    ftm = np.maximum(ftm, env._DIST_FLOOR)
    dropped = rng.random(dist.shape) < env.P_DROP_FTM
    avail = env.FTM_CAPABLE[None, :] & (~dropped)
    return ftm, avail


def generate_observations(rng: np.random.Generator, n: int) -> Observations:
    """Повний конвеєр генерації подій (позиції → RSSI → FTM)."""
    pos = generate_positions(rng, n)
    rssi, rssi_avail = observe_rssi(rng, pos)
    ftm, ftm_avail = observe_ftm(rng, pos)
    return Observations(
        pos=pos, rssi=rssi, rssi_avail=rssi_avail,
        ftm=ftm, ftm_avail=ftm_avail, in_crit=env.in_critical(pos),
    )

In [ ]:
%%writefile src/adversary.py
"""Модель супротивника: evil twin, deceptive ranging, deauthentication.

Adversary model for the localization pipeline.

Реалізує три класи навмисних впливів із Розділу 1:

* **evil_twin** — клонування довіри: атакуючий підвищує уявний RSSI обраного AP,
  щоб «підтягнути» RSSI-локалізацію до хибної точки (п. 1.1.4);
* **deceptive_ranging** — маніпуляція часовою дальнометрією: додатне/від'ємне
  зміщення FTM-дальності обраного AP (п. 1.3.8);
* **deauth** — атака на доступність: примусове відключення каналу AP (п. 1.1.5).

Кожна функція повертає *нову* копію :class:`~aegis_rf.sensing.Observations`, не
змінюючи вхідні дані. Індекси/зміщення атак — детерміновані параметри.
"""

from __future__ import annotations

import numpy as np

from .sensing import Observations


def apply_evil_twin(obs: Observations, ap_index: int, boost_db: float = 15.0,
                    fraction: float = 1.0, rng: np.random.Generator | None = None) -> Observations:
    """Підняти RSSI AP ``ap_index`` на ``boost_db`` дБ для частки подій ``fraction``."""
    out = obs.copy()
    mask = _event_mask(len(out.pos), fraction, rng)
    out.rssi[mask, ap_index] += boost_db
    out.rssi_avail[mask, ap_index] = True  # клон завжди «видно»
    return out


def apply_deceptive_ranging(obs: Observations, ap_index: int, bias_m: float = 8.0,
                            fraction: float = 1.0, rng: np.random.Generator | None = None) -> Observations:
    """Внести зміщення ``bias_m`` (м) у FTM-дальність AP ``ap_index``."""
    out = obs.copy()
    mask = _event_mask(len(out.pos), fraction, rng) & out.ftm_avail[:, ap_index]
    out.ftm[mask, ap_index] = np.maximum(out.ftm[mask, ap_index] + bias_m, 0.3)
    return out


def apply_deauth(obs: Observations, ap_index: int, fraction: float = 1.0,
                 rng: np.random.Generator | None = None) -> Observations:
    """Відключити RSSI та FTM AP ``ap_index`` для частки подій ``fraction``."""
    out = obs.copy()
    mask = _event_mask(len(out.pos), fraction, rng)
    out.rssi_avail[mask, ap_index] = False
    out.ftm_avail[mask, ap_index] = False
    return out


def _event_mask(n: int, fraction: float, rng: np.random.Generator | None) -> np.ndarray:
    """Маска подій, до яких застосовується атака."""
    if fraction >= 1.0:
        return np.ones(n, dtype=bool)
    if rng is None:
        # Детермінований префікс, якщо генератор не передано
        mask = np.zeros(n, dtype=bool)
        mask[: int(round(n * fraction))] = True
        return mask
    return rng.random(n) < fraction

In [ ]:
%%writefile src/localization.py
"""Байєсівська локалізація та гібридне злиття модальностей.

Bayesian localization and hybrid modality fusion.

Правдоподібність позиції ``p`` за спостереженнями будується як добуток гауссових
членів по доступних AP (п. 1.2.4, 1.4.4):

* RSSI:  -0.5 * ((s_a - M_a(p)) / sigma_rssi)^2
* FTM:   -0.5 * ((r_a - ||p - AP_a||) / sigma_ftm)^2

Оцінка позиції — MAP (argmax апостеріорної щільності на RP-сітці). Апостеріорна
ймовірність перебування у критичній зоні — сума нормованої постеріорної маси по
RP усередині зони.

Чотири методи:

* ``rssi``   — лише RSSI-радіокарта;
* ``ftm``    — лише FTM/RTT-дальнометрія;
* ``fusion`` — зважений добуток правдоподібностей обох модальностей;
* ``robust`` — те саме зі стійким відсіюванням AP-викидів (evil twin / deceptive
  ranging): AP зі стандартизованим залишком понад поріг у поточному MAP
  вимикається, оцінка перераховується (ітеративне переважування).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from . import environment as env
from .sensing import Observations

# Ваги модальностей у злитті (довіра до RSSI-покриття vs FTM-геометрії)
W_RSSI = 1.0
W_FTM = 1.4

# Поріг стандартизованого залишку для робастного відсіювання AP
ROBUST_GATE = 3.0
ROBUST_ITERS = 2


@dataclass
class LocResult:
    """Результат локалізації набору подій одним методом."""

    est: np.ndarray  # (E, 2) оцінені позиції
    p_crit: np.ndarray  # (E,) апостеріорна P(критична зона)
    error: np.ndarray  # (E,) похибка локалізації, м


def _accum(values: np.ndarray, avail: np.ndarray, ref: np.ndarray,
           sigma: float, weights: np.ndarray | None = None) -> np.ndarray:
    """Накопичити 0.5*((value-ref)/sigma)^2 по AP → матриця ``(E, G)``.

    ``values``/``avail`` — ``(E, A)``; ``ref`` — ``(G, A)`` (радіокарта M або
    відстані D). Недоступні AP не враховуються; ``weights`` (E, A) множать внесок.
    """
    e, a = values.shape
    g = ref.shape[0]
    acc = np.zeros((e, g))
    for k in range(a):
        m = avail[:, k]
        if not m.any():
            continue
        diff = (values[m, k][:, None] - ref[None, :, k]) / sigma
        contr = 0.5 * diff * diff
        if weights is not None:
            contr = contr * weights[m, k][:, None]
        acc[m] += contr
    return acc


def _nll(obs: Observations, radiomap, w_rssi: float, w_ftm: float,
         wr: np.ndarray | None = None, wf: np.ndarray | None = None,
         use_rssi: bool = True, use_ftm: bool = True) -> np.ndarray:
    """Негативна лог-правдоподібність на сітці для заданих модальностей."""
    _, m, d_grid, _ = radiomap
    e = len(obs.pos)
    nll = np.zeros((e, m.shape[0]))
    if use_rssi:
        nll += w_rssi * _accum(obs.rssi, obs.rssi_avail, m, env.RSSI_SIGMA, wr)
    if use_ftm:
        nll += w_ftm * _accum(obs.ftm, obs.ftm_avail, d_grid, env.FTM_SIGMA, wf)
    return nll


def _finish(nll: np.ndarray, obs: Observations, radiomap) -> LocResult:
    grid, _, _, crit_mask = radiomap
    idx = np.argmin(nll, axis=1)
    est = grid[idx]
    # Апостеріорна маса в критичній зоні (softmax зі стабілізацією)
    shifted = nll - nll.min(axis=1, keepdims=True)
    w = np.exp(-shifted)
    w /= w.sum(axis=1, keepdims=True)
    p_crit = w[:, crit_mask].sum(axis=1)
    error = np.sqrt(((est - obs.pos) ** 2).sum(axis=1))
    return LocResult(est=est, p_crit=p_crit, error=error)


def localize_rssi(obs: Observations, radiomap) -> LocResult:
    return _finish(_nll(obs, radiomap, W_RSSI, W_FTM, use_ftm=False), obs, radiomap)


def localize_ftm(obs: Observations, radiomap) -> LocResult:
    return _finish(_nll(obs, radiomap, W_RSSI, W_FTM, use_rssi=False), obs, radiomap)


def localize_fusion(obs: Observations, radiomap) -> LocResult:
    return _finish(_nll(obs, radiomap, W_RSSI, W_FTM), obs, radiomap)


def localize_robust(obs: Observations, radiomap) -> LocResult:
    """Гібридне злиття з ітеративним відсіюванням AP-викидів."""
    grid, m, d_grid, _ = radiomap
    wr = obs.rssi_avail.astype(float)
    wf = obs.ftm_avail.astype(float)

    nll = _nll(obs, radiomap, W_RSSI, W_FTM, wr=wr, wf=wf)
    for _ in range(ROBUST_ITERS):
        idx = np.argmin(nll, axis=1)
        # Стандартизовані залишки в поточному MAP
        m_at = m[idx]  # (E, N_AP)
        d_at = d_grid[idx]  # (E, N_AP)
        resid_r = np.abs(obs.rssi - m_at) / env.RSSI_SIGMA
        resid_f = np.abs(obs.ftm - d_at) / env.FTM_SIGMA
        wr = obs.rssi_avail.astype(float) * (resid_r <= ROBUST_GATE)
        wf = obs.ftm_avail.astype(float) * (resid_f <= ROBUST_GATE)
        # Захист від виродженого випадку: якщо все відсіяно — лишаємо доступне
        wr = _guard(wr, obs.rssi_avail)
        wf = _guard(wf, obs.ftm_avail)
        nll = _nll(obs, radiomap, W_RSSI, W_FTM, wr=wr, wf=wf)
    return _finish(nll, obs, radiomap)


def _guard(weights: np.ndarray, avail: np.ndarray) -> np.ndarray:
    """Не дозволяти повністю відсіяти всі AP події (лишаємо доступні як є)."""
    kept = weights.sum(axis=1)
    empty = kept == 0
    if empty.any():
        weights = weights.copy()
        weights[empty] = avail[empty].astype(float)
    return weights


METHODS = {
    "rssi": localize_rssi,
    "ftm": localize_ftm,
    "fusion": localize_fusion,
    "robust": localize_robust,
}


def localize(obs: Observations, radiomap, method: str) -> LocResult:
    """Локалізувати події заданим методом (``rssi`` | ``ftm`` | ``fusion`` | ``robust``)."""
    return METHODS[method](obs, radiomap)

In [ ]:
%%writefile src/metrics.py
"""Метрики точності локалізації та зонової атрибуції з bootstrap-CI.

Localization-accuracy and zone-attribution metrics with bootstrap CIs.

* Точність позиціонування: медіана, RMSE, P90 похибки (м);
* Зонова атрибуція (критична зона): точність, Pd (виявлення істинно-критичних),
  FAR (хибні тривоги на дозволених позиціях);
* 95 % CI медіанної похибки — percentile bootstrap.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from .localization import LocResult
from .sensing import Observations

N_BOOT = 600
CRIT_THRESHOLD = 0.5  # поріг рішення P(критична зона) → тривога


def localization_metrics(res: LocResult) -> dict:
    """Медіана, RMSE та P90 похибки локалізації (м)."""
    err = res.error
    return {
        "median_err_m": float(np.median(err)),
        "rmse_m": float(np.sqrt(np.mean(err**2))),
        "p90_err_m": float(np.percentile(err, 90)),
    }


def attribution_metrics(res: LocResult, obs: Observations) -> dict:
    """Метрики зонової атрибуції: accuracy, Pd, FAR."""
    pred = res.p_crit >= CRIT_THRESHOLD
    true = obs.in_crit
    pd_ = pred[true].mean() if true.any() else float("nan")
    far = pred[~true].mean() if (~true).any() else float("nan")
    return {
        "zone_acc": float((pred == true).mean()),
        "crit_pd": float(pd_),
        "crit_far": float(far),
    }


def bootstrap_median_ci(res: LocResult, seed: int, n_boot: int = N_BOOT):
    """95 % percentile-bootstrap CI медіанної похибки локалізації."""
    err = res.error
    n = len(err)
    rng = np.random.default_rng(seed)
    meds = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        meds[b] = np.median(err[idx])
    lo, hi = np.percentile(meds, [2.5, 97.5])
    return float(lo), float(hi)


def evaluate(res: LocResult, obs: Observations, boot_seed: int) -> dict:
    """Повний набір метрик для одного методу в одному сценарії."""
    row = {}
    row.update(localization_metrics(res))
    row.update(attribution_metrics(res, obs))
    lo, hi = bootstrap_median_ci(res, boot_seed)
    row["median_ci_lo"] = lo
    row["median_ci_hi"] = hi
    return row


def metrics_table(results: dict) -> pd.DataFrame:
    """Зібрати таблицю метрик з вкладеного словника ``{scenario: {method: row}}``."""
    rows = []
    for scenario, methods in results.items():
        for method, row in methods.items():
            rows.append({"scenario": scenario, "method": method, **row})
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/attribution.py
"""Spatial Attribution Record (SAR) — мінімальна одиниця доказу для SOC/SIEM.

Формує доказовий запис просторової атрибуції (п. 1.5.3): оцінена позиція,
невизначеність, апостеріорна ймовірність критичної зони, використані модальності,
ознаки цілісності (integrity flags) та провенанс. Придатний для відображення у
модель даних SIEM і подальшої SOAR-обробки (п. 1.5.4).

Записи *демонстраційні*; поля провенансу (seed, версія радіокарти) фіксують
відтворюваність, а не реальні часові мітки.
"""

from __future__ import annotations

import json

import numpy as np

from . import environment as env
from .localization import LocResult
from .sensing import Observations


def build_records(obs: Observations, res: LocResult, radiomap, *,
                  scenario: str, method: str, seed: int,
                  integrity_flags: np.ndarray | None = None,
                  indices=None) -> list[dict]:
    """Сформувати список SAR-записів для обраних подій ``indices`` (за замовч. усі)."""
    grid, _, _, _ = radiomap
    if indices is None:
        indices = range(len(obs.pos))

    records = []
    for i in indices:
        est = res.est[i]
        p_crit = float(res.p_crit[i])
        zone = "critical" if p_crit >= 0.5 else "allowed"
        flags = []
        if integrity_flags is not None and integrity_flags[i]:
            flags.append("modality_inconsistency")
        n_rssi = int(obs.rssi_avail[i].sum())
        n_ftm = int(obs.ftm_avail[i].sum())
        if n_rssi + n_ftm < 3:
            flags.append("sparse_observation")

        records.append(
            {
                "record_type": "SpatialAttributionRecord",
                "event_id": int(i),
                "estimate_xy_m": [round(float(est[0]), 3), round(float(est[1]), 3)],
                "position_error_m": round(float(res.error[i]), 3),
                "zone_posterior": {"critical": round(p_crit, 4),
                                    "allowed": round(1.0 - p_crit, 4)},
                "attributed_zone": zone,
                "modalities": {"rssi_aps": n_rssi, "ftm_aps": n_ftm},
                "integrity_flags": flags,
                "fusion_method": method,
                "provenance": {
                    "scenario": scenario,
                    "seed": int(seed),
                    "radiomap_grid_points": int(grid.shape[0]),
                    "n_ap": int(env.N_AP),
                },
            }
        )
    return records


def to_siem_json(records: list[dict]) -> str:
    """Серіалізувати SAR-записи у JSON (по одному об'єкту на рядок — SIEM-friendly)."""
    return "\n".join(json.dumps(r, ensure_ascii=False) for r in records)


def integrity_flags(obs: Observations, radiomap) -> np.ndarray:
    """Ознака неузгодженості модальностей: розбіжність RSSI- та FTM-оцінок понад поріг.

    Використовується як детектор навмисної маніпуляції (evil twin / deceptive
    ranging): якщо позиції за окремими модальностями розходяться більше, ніж на
    ``gate`` метрів, подія позначається для перевірки оператором.
    """
    from .localization import localize_ftm, localize_rssi

    gate = 5.0
    r = localize_rssi(obs, radiomap).est
    f = localize_ftm(obs, radiomap).est
    disagreement = np.sqrt(((r - f) ** 2).sum(axis=1))
    return disagreement > gate

In [ ]:
%%writefile src/make_figures.py
"""Побудова рисунків AEGIS-RF (inline, matplotlib).

Figure generation for the AEGIS-RF reproducibility package.

Кожна функція повертає :class:`matplotlib.figure.Figure`; за переданого ``save``
рисунок зберігається (320 dpi). Модуль не викликає ``plt.show()`` — це лишено
ноутбуку, тож функції придатні для тестів у безголовому режимі.
"""

from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from . import environment as env
from .localization import localize
from .sensing import Observations

METHOD_TITLES = {
    "rssi": "Лише RSSI",
    "ftm": "Лише FTM/RTT",
    "fusion": "Наївне злиття",
    "robust": "Робастне злиття",
}
METHOD_COLORS = {
    "rssi": "#777777",
    "ftm": "#6ACC65",
    "fusion": "#EE854A",
    "robust": "#4878CF",
}
SCENARIO_TITLES = {
    "clean": "Без атак",
    "evil_twin": "Evil twin",
    "deceptive_ranging": "Deceptive ranging",
    "deauth": "Deauth",
}


def setup_matplotlib() -> None:
    plt.rcParams.update(
        {
            "figure.dpi": 110, "savefig.dpi": 320, "font.size": 10,
            "axes.grid": True, "grid.alpha": 0.3,
            "axes.spines.top": False, "axes.spines.right": False,
        }
    )


def _save(fig, save):
    if save:
        fig.savefig(save, bbox_inches="tight")


def _draw_room(ax):
    """Намалювати контур приміщення, критичну зону та точки доступу."""
    ax.add_patch(plt.Rectangle((0, 0), env.AREA_W, env.AREA_H, fill=False, ec="#333", lw=1.2))
    ax.add_patch(
        plt.Rectangle(
            (env.CRIT_X0, env.CRIT_Y0), env.CRIT_X1 - env.CRIT_X0, env.CRIT_Y1 - env.CRIT_Y0,
            fc="#D65F5F", ec="#D65F5F", alpha=0.15, lw=1.2, label="критична зона",
        )
    )
    for i, (x, y) in enumerate(env.AP_POSITIONS):
        marker = "^" if env.FTM_CAPABLE[i] else "s"
        ax.plot(x, y, marker, ms=11, color="#1f3b73", mec="k", zorder=5)
        ax.annotate(f"AP{i}", (x, y), textcoords="offset points", xytext=(6, 5), fontsize=8)
    ax.set_xlim(-1, env.AREA_W + 1)
    ax.set_ylim(-1, env.AREA_H + 1)
    ax.set_aspect("equal")
    ax.set_xlabel("x, м")
    ax.set_ylabel("y, м")


# ------------------------------------------------------------------
# Рис. 1. Карта середовища КІІ
# ------------------------------------------------------------------
def fig_environment(exp, save=None):
    fig, ax = plt.subplots(figsize=(7.2, 6))
    _draw_room(ax)
    pos = exp.base_obs.pos
    ax.scatter(pos[:, 0], pos[:, 1], s=6, alpha=0.25, color="#4878CF", label="події (джерела)")
    ax.set_title("Рис. 1. Середовище КІІ: AP (▲ = FTM), критична зона, події")
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 2. CDF похибки локалізації (сценарій clean)
# ------------------------------------------------------------------
def fig_error_cdf(exp, scenario="clean", save=None):
    obs = exp.scenarios[scenario]
    fig = plt.figure(figsize=(7, 4.2))
    for method in ("rssi", "ftm", "fusion", "robust"):
        err = np.sort(localize(obs, exp.radiomap, method).error)
        cdf = np.linspace(0, 1, len(err))
        plt.plot(err, cdf, color=METHOD_COLORS[method], label=METHOD_TITLES[method])
    plt.xlim(0, 12)
    plt.xlabel("Похибка локалізації, м")
    plt.ylabel("F(x)")
    plt.title(f"Рис. 2. CDF похибки локалізації ({SCENARIO_TITLES[scenario]})")
    plt.legend()
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 3. Медіанна похибка за сценаріями та методами
# ------------------------------------------------------------------
def fig_error_bars(table, save=None):
    scenarios = list(SCENARIO_TITLES)
    methods = ["rssi", "ftm", "fusion", "robust"]
    fig, ax = plt.subplots(figsize=(9.5, 4.4))
    width = 0.2
    xs = np.arange(len(scenarios))
    for j, method in enumerate(methods):
        vals, los, his = [], [], []
        for sc in scenarios:
            row = table[(table.scenario == sc) & (table.method == method)].iloc[0]
            vals.append(row.median_err_m)
            los.append(row.median_err_m - row.median_ci_lo)
            his.append(row.median_ci_hi - row.median_err_m)
        ax.bar(xs + (j - 1.5) * width, vals, width, yerr=[los, his], capsize=3,
               color=METHOD_COLORS[method], label=METHOD_TITLES[method], alpha=0.9)
    ax.set_xticks(xs)
    ax.set_xticklabels([SCENARIO_TITLES[s] for s in scenarios])
    ax.set_ylabel("Медіанна похибка, м (95 % CI)")
    ax.set_title("Рис. 3. Точність локалізації за сценаріями атак")
    ax.legend(ncol=4, fontsize=8)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 4. Стійкість атрибуції: Pd / FAR критичної зони
# ------------------------------------------------------------------
def fig_attribution(table, save=None):
    scenarios = list(SCENARIO_TITLES)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (col, ttl) in zip(axes, [("crit_pd", "Pd критичної зони"),
                                     ("crit_far", "FAR критичної зони")]):
        xs = np.arange(len(scenarios))
        width = 0.2
        for j, method in enumerate(["rssi", "ftm", "fusion", "robust"]):
            vals = [table[(table.scenario == sc) & (table.method == method)].iloc[0][col]
                    for sc in scenarios]
            ax.bar(xs + (j - 1.5) * width, vals, width,
                   color=METHOD_COLORS[method], label=METHOD_TITLES[method], alpha=0.9)
        ax.set_xticks(xs)
        ax.set_xticklabels([SCENARIO_TITLES[s] for s in scenarios], rotation=15)
        ax.set_ylim(0, 1.02)
        ax.set_title(ttl)
    axes[0].legend(ncol=2, fontsize=8)
    fig.suptitle("Рис. 4. Зонова атрибуція: виявлення критичної зони під атаками", y=1.03)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 5. Просторовий ефект атаки: зміщення оцінок (evil twin)
# ------------------------------------------------------------------
def fig_attack_shift(exp, scenario="evil_twin", n_show=120, save=None):
    obs = exp.scenarios[scenario]
    naive = localize(obs, exp.radiomap, "fusion").est
    robust = localize(obs, exp.radiomap, "robust").est
    true = obs.pos
    rng_idx = np.linspace(0, len(true) - 1, min(n_show, len(true))).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    for ax, est, ttl in [(axes[0], naive, "Наївне злиття"), (axes[1], robust, "Робастне злиття")]:
        _draw_room(ax)
        for i in rng_idx:
            ax.plot([true[i, 0], est[i, 0]], [true[i, 1], est[i, 1]],
                    "-", color="#999999", lw=0.4, alpha=0.6)
        ax.scatter(true[rng_idx, 0], true[rng_idx, 1], s=10, color="#4878CF", label="істина", zorder=4)
        ax.scatter(est[rng_idx, 0], est[rng_idx, 1], s=10, color="#D65F5F", label="оцінка", zorder=4)
        ax.set_title(ttl)
        ax.legend(loc="upper left", fontsize=8)
    fig.suptitle(f"Рис. 5. Ефект атаки «{SCENARIO_TITLES[scenario]}»: наївне vs робастне злиття", y=1.02)
    fig.tight_layout()
    _save(fig, save)
    return fig

In [ ]:
%%writefile src/pipeline.py
"""Оркестрація експерименту: сценарії × методи → таблиця метрик.

Experiment orchestration: scenarios x methods -> metrics table.

Сценарії:

* ``clean``             — без атак (базова точність);
* ``evil_twin``         — RSSI обраного AP підвищено (клонування довіри);
* ``deceptive_ranging`` — FTM-дальність обраного AP зміщено;
* ``deauth``            — обраний AP відключено (атака на доступність).

Для кожного сценарію обчислюються всі чотири методи локалізації. Уся випадковість
походить з єдиного генератора ``rng(SEED)`` у фіксованому порядку, тож результати
відтворюються біт-у-біт.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from . import adversary, environment as env
from .localization import METHODS, localize
from .metrics import evaluate, metrics_table
from .sensing import Observations, generate_observations

SEED = 80211  # зерно відтворюваності (натяк на IEEE 802.11)
N_EVENTS = 1500

# Параметри атак (детерміновані)
EVIL_TWIN_AP = 4  # центральний AP (15, 12)
EVIL_TWIN_BOOST_DB = 15.0
DECEPTIVE_AP = 5  # FTM-сумісний AP біля критичної зони (24, 18)
DECEPTIVE_BIAS_M = 8.0
DEAUTH_AP = 3  # кутовий AP (28, 22)

SCENARIOS = ["clean", "evil_twin", "deceptive_ranging", "deauth"]


@dataclass
class Experiment:
    radiomap: tuple
    base_obs: Observations
    scenarios: dict  # {name: Observations}


def build_scenarios(rng: np.random.Generator, base: Observations) -> dict:
    """Побудувати спостереження для кожного сценарію з базового набору."""
    return {
        "clean": base,
        "evil_twin": adversary.apply_evil_twin(base, EVIL_TWIN_AP, EVIL_TWIN_BOOST_DB),
        "deceptive_ranging": adversary.apply_deceptive_ranging(base, DECEPTIVE_AP, DECEPTIVE_BIAS_M),
        "deauth": adversary.apply_deauth(base, DEAUTH_AP),
    }


def run_experiment(seed: int = SEED, n_events: int = N_EVENTS) -> Experiment:
    """Згенерувати середовище, події та всі сценарії атак."""
    radiomap = env.build_radiomap()
    rng = np.random.default_rng(seed)
    base = generate_observations(rng, n_events)
    scenarios = build_scenarios(rng, base)
    return Experiment(radiomap=radiomap, base_obs=base, scenarios=scenarios)


def compute_results(exp: Experiment, boot_seed: int = SEED + 1) -> dict:
    """Обчислити метрики для всіх сценаріїв і методів → ``{scenario: {method: row}}``."""
    results = {}
    for sc_name, obs in exp.scenarios.items():
        results[sc_name] = {}
        for method in METHODS:
            res = localize(obs, exp.radiomap, method)
            results[sc_name][method] = evaluate(res, obs, boot_seed)
    return results


def run(seed: int = SEED, n_events: int = N_EVENTS):
    """Повний прогін: повертає ``(experiment, results_dict, metrics_dataframe)``."""
    exp = run_experiment(seed, n_events)
    results = compute_results(exp)
    table = metrics_table(results)
    return exp, results, table

Пакет записано. Далі — звичайний імпорт і оркестрація.

In [ ]:
# ============================================================
# Налаштування / Setup (самодостатня версія)
# ============================================================
import os, json, hashlib, warnings
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import src.pipeline as P
import src.make_figures as figs

warnings.filterwarnings("ignore")
figs.setup_matplotlib()
for d in ("results", "figures"):
    os.makedirs(d, exist_ok=True)

print("SEED:", P.SEED, "| events:", P.N_EVENTS, "| scenarios:", P.SCENARIOS)
print("NumPy:", np.__version__)

## 1. Середовище КІІ / CII environment

Синтетичне приміщення $30\times24$ м із шістьма точками доступу IEEE 802.11
(частина підтримує FTM/RTT), виділеною **критичною зоною** (напр. серверна) та
регулярною RP-сіткою радіокарти. Джерела сигналу (події) рівномірно розподілені,
близько 45 % — усередині критичної зони.

In [ ]:
exp = P.run_experiment()
figs.fig_environment(exp, save="figures/fig1_environment.png"); plt.show()

## 2. Локалізація та злиття модальностей / Localization & fusion

Байєсівська локалізація на RP-сітці: правдоподібність позиції — добуток гауссових
членів по доступних AP (RSSI-радіокарта та FTM-дальнометрія). Порівнюємо чотири методи:

* **RSSI** — лише радіокарта;
* **FTM/RTT** — лише часова дальнометрія;
* **Наївне злиття** — зважений добуток правдоподібностей;
* **Робастне злиття** — те саме з ітеративним відсіюванням AP-викидів (стійкість
  до evil twin / deceptive ranging).

Проганяємо всі методи на 4 сценаріях (без атак, evil twin, deceptive ranging, deauth)
і зводимо метрики в таблицю.

In [ ]:
exp, results, table = P.run()
cols = ["scenario","method","median_err_m","rmse_m","p90_err_m","zone_acc","crit_pd","crit_far"]
table.to_csv("results/metrics.csv", index=False)
table[cols].round(4)

### Інтерпретація

* **Без атак** гібридне злиття точніше за окремі модальності (менша медіанна похибка).
* **Evil twin** підриває RSSI-локалізацію; FTM лишається чистим, а робастне злиття
  відсіює підроблений AP і зберігає точність.
* **Deceptive ranging** «підтягує» наївне злиття (FTM-канал скомпрометовано), але
  робастний метод відновлює позицію.
* **Deauth** — плавна деградація: злиття зберігає працездатність за втрати одного AP.

In [ ]:
pivot = table.pivot(index="scenario", columns="method", values="median_err_m")[
    ["rssi","ftm","fusion","robust"]].reindex(P.SCENARIOS)
display(pivot.round(3))
figs.fig_error_bars(table, save="figures/fig3_error_bars.png"); plt.show()

## 3. Розподіл похибки / Error CDF

Емпіричні функції розподілу похибки локалізації (сценарій без атак): робастне та
наївне злиття домінують над окремими модальностями на всьому діапазоні.

In [ ]:
figs.fig_error_cdf(exp, scenario="clean", save="figures/fig2_error_cdf.png"); plt.show()

## 4. Зонова атрибуція / Zone attribution

Апостеріорна ймовірність перебування джерела в критичній зоні → рішення тривоги.
**Pd** — виявлення істинно-критичних подій, **FAR** — хибні тривоги на дозволених
позиціях. Робастне злиття утримує Pd під атаками, коли скомпрометована модальність
завалює наївні методи.

In [ ]:
figs.fig_attribution(table, save="figures/fig4_attribution.png"); plt.show()

## 5. Просторовий ефект атаки / Spatial effect of the attack

Лінії з'єднують істинну позицію з оцінкою. За «evil twin» наївне злиття системно
зміщує оцінки до підробленого AP; робастний метод відсіює викид і повертає їх на місце.

In [ ]:
figs.fig_attack_shift(exp, scenario="evil_twin", save="figures/fig5_attack_shift.png"); plt.show()

## 6. Spatial Attribution Record (SAR) для SOC/SIEM

Мінімальна одиниця доказу (п. 1.5.3): оцінена позиція, невизначеність, апостеріорна
ймовірність зони, використані модальності, ознаки цілісності (`integrity_flags`) та
провенанс. Формат — JSON по об'єкту на рядок, придатний для SIEM.

In [ ]:
from src.attribution import build_records, to_siem_json, integrity_flags
from src.localization import localize

obs = exp.scenarios["evil_twin"]
res = localize(obs, exp.radiomap, "robust")
flags = integrity_flags(obs, exp.radiomap)
records = build_records(obs, res, exp.radiomap, scenario="evil_twin", method="robust",
                        seed=P.SEED, integrity_flags=flags, indices=range(len(obs.pos)))
with open("results/spatial_attribution_records.jsonl", "w", encoding="utf-8") as fh:
    fh.write(to_siem_json(records))
print(f"Сформовано {len(records)} SAR-записів | подій з ознакою неузгодженості: {int(flags.sum())}")
print("\nПриклад запису:")
import json; print(json.dumps(records[0], ensure_ascii=False, indent=2))

## 7. Артефакти та контрольні суми / Artifacts & checksums

Таблиці — в `results/`, рисунки (320 dpi) — у `figures/`, доказові записи — у
`results/spatial_attribution_records.jsonl`. SHA-256 фіксують точний вміст для депозиту.

In [ ]:
manifest = {}
for root in ("results", "figures"):
    for f in sorted(os.listdir(root)):
        p = os.path.join(root, f)
        manifest[p] = hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
with open("results/MANIFEST_sha256.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
for p, h in manifest.items():
    print(f"{h}  {p}  ({os.path.getsize(p)/1024:.1f} КБ)")

## 8. Обмеження / Limitations

1. **Синтетичні дані.** Модель загасання, похибки FTM, частки NLOS та відмов задані
   параметрично, а не з польових вимірювань.
2. **Спрощена геометрія.** Один поверх, прямокутна критична зона, без стін/матеріалів
   і повного багатопроменевого моделювання.
3. **Модель супротивника** обмежена трьома класами атак на один AP; коаліційні та
   адаптивні атаки не розглядаються.
4. **Немає прямого емпіричного порівняння** з польовими системами.
5. Наведені числа **не можна використовувати** для сертифікації чи проєктування
   реальних систем захисту КІІ без валідації на реальних даних.

Ліцензія: MIT. Цитування — `CITATION.cff`. Прикладний проєкт за Розділом 1.